# **Desarrollo del taller 2 — Mini-Proyecto de clasificación de texto con Transformers**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yeavil-neny/NLP-ProcesamientoLeguajeNatural/blob/main/Taller_2/proyecto_transformers_clasificacion.ipynb)

**Grupo:** Delta

## **Definición del Problema y Carga de Datos**

**🎯 Objetivo del notebook:** resolver el **mismo problema del Taller 1** (clasificar la satisfacción de clientes en e-commerce a partir de reseñas en español, en una escala de 1 a 5 estrellas) pero **reemplazando la LSTM por un Transformer implementado desde cero**, tal como vimos en el notebook guía `0-transformers-from-scratch-comentado.ipynb` (que corresponde al `1-transformers-from-scratch` de la clase).

En el Taller 1 abordamos la **clasificación de la satisfacción de clientes en plataformas de e-commerce**: las plataformas reciben miles de reseñas imposibles de analizar manualmente, por lo que necesitamos un sistema que interprete automáticamente el sentimiento de cada reseña y lo clasifique en una escala de **1 a 5 estrellas**. Los datos provienen del dataset **`SetFit/amazon_reviews_multi_es`** (reseñas reales de compradores en Hugging Face), con sus desafíos: errores ortográficos, lenguaje informal y textos de longitudes variadas.

### **¿Por qué un Transformer para este problema?**

1. **La atención es directa y global:** en la LSTM la información viaja token a token, y el sentimiento de una reseña suele depender de palabras muy específicas ("terrible", "llegó roto", "excelente", "recomendado") o de contrastes ("se demoró mucho, **pero** la calidad es excelente"). La atención permite que cualquier palabra "mire" directamente a cualquier otra, sin importar la distancia.
2. **Es paralelizable:** a diferencia de la recurrencia, todas las posiciones se procesan a la vez, lo que acelera el entrenamiento en GPU.
3. **Es el estado del arte** en NLP: comprenderlo desde cero nos prepara para los modelos pre-entrenados (BERT, etc.) de las siguientes sesiones.

Los retos particulares de las reseñas para un Transformer: son **textos cortos** (frente a las noticias largas del guía) y **con 5 clases ordenadas** (1→5 estrellas), donde la confusión más probable está entre estrellas vecinas.

In [ ]:
# Setup: detectamos el entorno y silenciamos avisos
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("¿Ejecutando en Google Colab?", IN_COLAB)

In [ ]:
if IN_COLAB:
    !pip install -q datasets transformers tokenizers pytorch_lightning torchmetrics wordcloud

In [ ]:
# Importaciones principales
import math
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from datasets import load_dataset
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import pytorch_lightning as pl
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torchmetrics import Accuracy

# Reproducibilidad: fijamos semillas en PyTorch, numpy y Python
pl.seed_everything(42)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

### **Carga de Datos**

**🎯 Objetivo de la sección:** cargar el mismo dataset del Taller 1 (`SetFit/amazon_reviews_multi_es`) con el mismo muestreo (50.000 train / 5.000 val / 5.000 test, ya barajados con semilla 42) para que la comparación con los modelos LSTM sea **justa: mismos datos, misma partición, distinta arquitectura**.

In [ ]:
# Cargamos el dataset de reseñas de Amazon en español (mismo del Taller 1)
print("Descargando el dataset 'SetFit/amazon_reviews_multi_es'...")
dataset = load_dataset("SetFit/amazon_reviews_multi_es")

# Mismo submuestreo del Taller 1 (barajado con semilla fija para reproducibilidad)
subset_size_train = 50000
subset_size_eval = 5000

train_data = dataset['train'].shuffle(seed=42).select(range(subset_size_train))
val_data = dataset['validation'].shuffle(seed=42).select(range(subset_size_eval))
test_data = dataset['test'].shuffle(seed=42).select(range(subset_size_eval))

print(f"\n--- Resumen de Datos ---")
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print("\n--- Ejemplo de un registro ---")
print(f"Texto: {train_data[0]['text'][:250]}...")
print(f"Etiqueta (Estrellas): {train_data[0]['label']} (Escala de 0 a 4, donde 0 es 1 estrella)")

### **Análisis Exploratorio de Datos (EDA)**

**🎯 Objetivo de la sección:** re-analizar los datos (reutilizamos y ampliamos el EDA del Taller 1) para confirmar las decisiones de modelado en este nuevo contexto: balance de clases, longitud de las reseñas y vocabulario del sentimiento.

In [ ]:
# Convertimos el dataset a DataFrame para explorarlo con pandas
df_train = train_data.to_pandas()
df_train.head(10)

In [ ]:
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud

# Stopwords en español para limpiar las nubes de palabras
nltk.download('stopwords', quiet=True)
stop_words_sp = set(stopwords.words('spanish'))
stop_words_sp.update(['producto', 'amazon', 'bien', 'mal', 'hace', 'solo', 'si', 'mas', 'q', 'que'])

df_train['word_count'] = df_train['text'].apply(lambda x: len(str(x).split()))

# --- GRÁFICO 1: Distribución de Clases ---
plt.figure(figsize=(8, 5))
sns.countplot(data=df_train, x='label', palette='viridis')
plt.title('Distribución de Reseñas por Nivel de Satisfacción\n(0 = 1 Estrella ... 4 = 5 Estrellas)')
plt.xlabel('Etiqueta (Label)')
plt.ylabel('Cantidad de Reseñas')
plt.show()

# --- GRÁFICO 2: Longitud de Textos (extremos 1 y 5 estrellas) ---
plt.figure(figsize=(10, 5))
sns.histplot(data=df_train[df_train['label'] == 0], x='word_count', color='red', label='1 Estrella (Label 0)', kde=True, alpha=0.5)
sns.histplot(data=df_train[df_train['label'] == 4], x='word_count', color='green', label='5 Estrellas (Label 4)', kde=True, alpha=0.5)
plt.title('Distribución de la Longitud de las Reseñas (Cantidad de Palabras)')
plt.xlabel('Cantidad de Palabras')
plt.ylabel('Frecuencia')
plt.legend()
plt.xlim(0, 100)
plt.show()

# --- GRÁFICO 3: Nubes de Palabras (extremos de satisfacción) ---
text_negativo = " ".join(df_train[df_train['label'] == 0]['text'].tolist())
text_positivo = " ".join(df_train[df_train['label'] == 4]['text'].tolist())

wc_neg = WordCloud(stopwords=stop_words_sp, background_color='white', colormap='Reds', width=400, height=300).generate(text_negativo)
wc_pos = WordCloud(stopwords=stop_words_sp, background_color='white', colormap='Greens', width=400, height=300).generate(text_positivo)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(wc_neg, interpolation='bilinear')
axes[0].set_title('Palabras Frecuentes - 1 Estrella (Insatisfecho)')
axes[0].axis('off')
axes[1].imshow(wc_pos, interpolation='bilinear')
axes[1].set_title('Palabras Frecuentes - 5 Estrellas (Satisfecho)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

**Observaciones del EDA (revisadas del Taller 1):**

- **Balance de clases:** las 5 categorías están prácticamente igual representadas en el dataset original, por lo que la *accuracy* es una métrica razonable (aunque la complementaremos con el reporte de clasificación y la matriz de confusión).
- **Textos cortos:** la gran mayoría de las reseñas tiene menos de ~75 palabras, muy distinto del corpus de noticias del notebook guía (que requería `max_len=2048`). Esto juega a favor del Transformer: secuencias cortas hacen el costo cuadrático de la atención muy manejable.
- **Separación semántica clara en los extremos:** las nubes de palabras muestran vocabulario directamente asociado al sentimiento; sin embargo, las clases intermedias (2 y 3 estrellas) mezclan pros y contras en un mismo texto, y ahí es donde esperamos que se concentren los errores (igual que ocurría con la LSTM).

In [ ]:
# Ejemplos textuales por categoría (0-4 -> 1-5 estrellas)
for estrella in range(5):
    print("=" * 50)
    print(f"CATEGORÍA {estrella} ({estrella + 1} Estrellas)")
    print("=" * 50)
    ejemplos = df_train[df_train['label'] == estrella].head(2)
    for idx, row in ejemplos.iterrows():
        print(f" {row['text']}\n")

### **Tokenización: BPE entrenado sobre las reseñas**

**🎯 Objetivo de la sección:** entrenar un tokenizador **Byte-Pair Encoding (BPE)** propio sobre nuestro corpus de reseñas, como en el notebook guía. A diferencia del vocabulario *word-level* del Taller 1 (20.000 palabras + `[UNK]`), el BPE trabaja con **subpalabras**: nunca produce tokens "desconocidos" (no hay `[UNK]`), es robusto a errores ortográficos y tildes, y genera tokens más informativos que caracteres sueltos.

In [ ]:
from tokenizers.pre_tokenizers import ByteLevel
from transformers import AutoTokenizer

# Base: el tokenizador de GPT-2 (misma maquinaria BPE), re-entrenado desde cero
base_tokenizer = AutoTokenizer.from_pretrained("gpt2")
base_vocab = list(ByteLevel.alphabet())  # alfabeto byte-level completo

# Iterador que entrega los textos de TRAIN en lotes
# (entrenamos solo con train: val/test nunca participan en la construcción del vocabulario)
def batch_iterator(batch_size=1000):
    for i in range(0, len(train_data), batch_size):
        yield train_data[i:i + batch_size]['text']

# Entrenamiento del tokenizador BPE:
# - vocab_size 30.000: las reseñas son cortas y el corpus es más "homogéneo" que noticias,
#   por lo que 30k cubre de sobra (en el Taller 1 ya vimos ~20k cubría bien este dominio)
# - new_special_tokens: añadimos [PAD] para el relleno de las secuencias
reviews_tokenizer = base_tokenizer.train_new_from_iterator(
    batch_iterator(),
    vocab_size=30000,
    initial_alphabet=base_vocab,
    new_special_tokens=["[PAD]"],
    show_progress=True,
)

if reviews_tokenizer.pad_token is None:
    reviews_tokenizer.pad_token = '[PAD]'

print(f"Vocabulario final: {len(reviews_tokenizer)} tokens")
print(f"ID del token de relleno: {reviews_tokenizer.pad_token_id}")

In [ ]:
# Probamos el tokenizador con una reseña típica del dominio
ejemplo = "¡Este producto es terrible! Llegó tarde y roto. No lo recomiendo."

# Los tokens como piezas de texto (Ġ indica un espacio precediendo al token)
tokens_str = reviews_tokenizer.tokenize(ejemplo)
print("Tokens:", tokens_str)

# Y la salida estructurada: ids + máscara de atención (longitud fija de 16)
result = reviews_tokenizer(ejemplo, max_length=16, truncation=True, padding='max_length')
print("input_ids:      ", result['input_ids'])
print("attention_mask: ", result['attention_mask'])

In [ ]:
# Exploramos el vocabulario aprendido (ordenado por id = orden de fusión BPE)
vocab = reviews_tokenizer.get_vocab()
tokens_sorted = sorted(vocab.items(), key=lambda x: x[1])
print("Primeros 15 tokens:", [reviews_tokenizer.convert_tokens_to_string([t]) for t, _ in tokens_sorted[:15]])
print("15 del medio:      ", [reviews_tokenizer.convert_tokens_to_string([t]) for t, _ in tokens_sorted[1000:1015]])
print("Últimos 15 tokens: ", [reviews_tokenizer.convert_tokens_to_string([t]) for t, _ in tokens_sorted[-15:]])

**Análisis del vocabulario:** al igual que en el notebook guía, los primeros tokens son los caracteres especiales y la puntuación (muy relevante aquí: los signos `¡ !` son portadores de sentimiento en reseñas), en el medio aparecen las partes de palabras más frecuentes del corpus y al final los tokens raros. Los tokens de en medio deberían mostrar fragmentos típicos del dominio e-commerce ("producto", "calidad", "entrega", "llegó", etc.); vale la pena revisar la salida y confirmarlo.

### **Eligiendo la longitud máxima de secuencia (max_len)**

En el Taller 1 fijamos `max_length=75` **palabras** con la curva de cobertura léxica. Aquí repetimos el análisis pero en **tokens BPE**: como el BPE parte palabras poco frecuentes en subpalabras, una reseña de 75 palabras suele necesitar *más* de 75 tokens.

In [ ]:
# Distribución de longitudes en TOKENS BPE sobre una muestra del train set
muestra = train_data.shuffle(seed=42).select(range(20000))

lens = []
for text in tqdm(muestra['text'], desc="Tokenizando muestra"):
    lens.append(len(reviews_tokenizer(text)['input_ids']))
lens = np.array(lens)

print(f"Media: {lens.mean():.1f} | Mediana: {np.median(lens):.0f} | P95: {np.percentile(lens, 95):.0f} | P99: {np.percentile(lens, 99):.0f} | Máx: {lens.max()}")

for candidate in (64, 96, 128, 160):
    cobertura = (lens <= candidate).mean() * 100
    print(f"max_len = {candidate:4d} tokens -> cubre el {cobertura:.2f}% de las reseñas")

**Decisión:** fijaremos `MAX_LEN = 128` tokens. Con ese valor se cubre el corpus casi por completo (revisa el P95/P99 de la celda anterior) sin desperdiciar cómputo: en secuencias cortas la atención cuadrática es barata. Un `MAX_LEN` mayor solo agregaría tokens de padding, que la máscara de atención ignora.

### **Dataset de PyTorch y DataLoaders**

**🎯 Objetivo de la sección:** envolver el dataset de HuggingFace en un `Dataset` de PyTorch que entregue tensores de longitud fija: `input_ids` (tokens), `attention_mask` (1 = token real, 0 = padding) y `y` (etiqueta). **Novedad respecto al Taller 1:** aquí la máscara de atención viaja junto a los datos, y será la que use el Transformer para ignorar el padding (en lugar del `padding_idx` / `pack_padded_sequence` de la LSTM).

In [ ]:
MAX_LEN = 128
NUM_CLASSES = 5  # reseñas de 1 a 5 estrellas (labels 0-4)

class AmazonReviewsTorchDataset(Dataset):
    """Dataset de PyTorch sobre el dataset de HuggingFace.

    Cada reseña se convierte en un diccionario de tensores:
    - input_ids: ids de los tokens (con padding hasta MAX_LEN)
    - attention_mask: 1 donde hay token real, 0 donde hay padding
    - y: id de la clase (0..4)
    """

    def __init__(self, hf_dataset, tokenizer, seq_length: int = 128):
        self.hf_dataset = hf_dataset
        self.tokenizer = tokenizer
        self.seq_length = seq_length

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, index):
        row = self.hf_dataset[index]
        # Tokenizamos con truncamiento/padding a longitud fija
        encoded = self.tokenizer(row['text'], max_length=self.seq_length, truncation=True, padding='max_length')
        data = {k: torch.tensor(v) for k, v in encoded.items()}
        # La etiqueta ya viene como id 0..4 (el dataset de Amazon ya es consecutivo)
        data['y'] = torch.tensor(row['label'])
        return data

# Instanciamos los tres conjuntos con el MISMO muestreo del Taller 1
train_dataset_pt = AmazonReviewsTorchDataset(train_data, reviews_tokenizer, MAX_LEN)
val_dataset_pt   = AmazonReviewsTorchDataset(val_data,   reviews_tokenizer, MAX_LEN)
test_dataset_pt  = AmazonReviewsTorchDataset(test_data,  reviews_tokenizer, MAX_LEN)

batch_size = 32 if torch.cuda.is_available() else 8
print(f"Batch size: {batch_size}")

train_loader = DataLoader(train_dataset_pt, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset_pt,   batch_size=batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset_pt,  batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Lotes por época (train): {len(train_loader)}")

In [ ]:
# Inspección de un batch
batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items()})
print("Ejemplo input_ids:", batch['input_ids'][0][:20].tolist())
print("Ejemplo attention_mask:", batch['attention_mask'][0][:20].tolist())

## **Positional Embeddings**

**🎯 Objetivo de la sección:** entender por qué la atención, por sí sola, es invariante al orden de las palabras (ve el texto como una "bolsa" de vectores) y cómo el *positional encoding* sinusoidal inyecta la información de posición sin agregar parámetros entrenables. Implementamos también la variante **aprendida** (como BERT/GPT) para poder experimentar con ambas.

Según el paper *Attention is All You Need*:

$$
PE(pos, 2i) = \sin(pos/10000^{2i/d_{model}}) \\
PE(pos, 2i + 1) = \cos(pos/10000^{2i/d_{model}})
$$

- $pos$ es la posición del token en la secuencia.
- $i$ es la dimensión del embedding.
- $d_{model}$ es la dimensionalidad total del embedding.

In [ ]:
class PosEncodingType:
    SINUSOID = 'sinusoid'
    LEARNABLE = 'learnable'


class SinusoidPE(nn.Module):
    """Positional Encoding sinusoidal, tal como lo define el paper.

    Se pre-calcula una sola vez (no tiene parámetros entrenables) y se suma a
    los embeddings de tokens: dimensiones pares llevan seno, impares coseno.
    """

    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        # Vector columna con las posiciones (pos) y vector fila con las dimensiones (i)
        pos = torch.arange(max_len).unsqueeze(1)
        i = torch.arange(d_model).unsqueeze(0)

        # Denominador 10000^(2i/d_model), aplicado a las posiciones
        div_term = 1 / torch.pow(10000, (2 * (i // 2)) / torch.tensor(d_model, dtype=torch.float32))
        angle_rads = pos * div_term

        pos_encoding = torch.zeros(max_len, d_model)
        pos_encoding[:, 0::2] = torch.sin(angle_rads[:, 0::2])  # pares  -> seno
        pos_encoding[:, 1::2] = torch.cos(angle_rads[:, 1::2])  # impares -> coseno

        # register_buffer: viaja con el modelo (también al GPU) pero NO se entrena
        self.register_buffer("pos_encoding", pos_encoding.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Sumamos el PE precalculado, truncado a la longitud real de la secuencia
        return x + self.pos_encoding[:, :x.size(1), :]


class LearnablePE(nn.Module):
    """Variante alternativa: PE aprendido.

    Una tabla nn.Embedding aprende un vector por posición durante el
    entrenamiento (el enfoque de BERT/GPT), con costo de parámetros y
    capacidad de adaptarse al dominio.
    """

    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(x.size(1), device=x.device)
        return x + self.embedding(positions).unsqueeze(0)


class TokenAndPosEmbedding(nn.Module):
    """Primera capa del modelo: ids de tokens -> vectores densos con info de posición.

    Combina un embedding de tokens (aprendido, con padding_idx para que los
    tokens de relleno tengan vector nulo) más el encoding posicional elegido.
    """

    def __init__(self, vocab_size: int, embed_dim: int, max_len: int,
                 pos_type: str = PosEncodingType.SINUSOID, pad_token_id: int = 0):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token_id)
        if pos_type == PosEncodingType.SINUSOID:
            self.pos_emb = SinusoidPE(max_len, embed_dim)
        else:
            self.pos_emb = LearnablePE(max_len, embed_dim)

    def forward(self, x):
        # 1) Embedding denso de cada token id
        token_emb = self.token_emb(x)
        # 2) Le sumamos el encoding posicional
        return self.pos_emb(token_emb)

In [ ]:
# Visualización del PE sinusoidal: cada fila es una posición, cada columna una dimensión
EMB_DIM = 256 if IN_COLAB else 128

tpe = TokenAndPosEmbedding(len(reviews_tokenizer), EMB_DIM, MAX_LEN)
pos_encoding = tpe.pos_emb.pos_encoding.squeeze(0).numpy()

plt.figure(figsize=(12, 6))
plt.pcolormesh(pos_encoding, cmap='viridis')
plt.xlabel('Dimensión del embedding')
plt.xlim((0, EMB_DIM))
plt.ylabel('Posición')
plt.colorbar()
plt.title('Positional Encoding sinusoidal')
plt.show()

Se observa la oscilación seno/coseno cambiando de "frecuencia" a lo largo de las dimensiones: las primeras dimensiones oscilan rápido (codifican posiciones finas) y las últimas lentamente (codifican posiciones gruesas). Cada posición recibe un patrón único, y las posiciones cercanas tienen patrones similares: de ahí la hipótesis de los autores de que el modelo puede aprender relaciones *relativas* entre posiciones.

## **Multi-Head Attention (el núcleo del Transformer)**

**🎯 Objetivo de la sección:** implementar la *Scaled Dot-Product Attention* y su extensión a múltiples cabezas, entendiendo el papel de Q (qué busco), K (qué ofrezco) y V (qué entrego), y el papel de la máscara de padding.

$$
	ext{Attention}(Q, K, V) = 	ext{softmax}\left(rac{QK^T}{\sqrt{d_K}}\right)V
$$

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention implementada desde cero.

    Mejoras respecto al notebook guía (documentadas para el taller):
    - assert con '%' (divisibilidad exacta) en vez del '&' bit a bit del ejemplo.
    - los parámetros usan 'embed_dim' (no una variable global).
    - la máscara de padding llena con -1e9 (numéricamente seguro), no con -9e-15.
    """

    def __init__(self, embed_dim: int, num_heads: int = 8):
        super().__init__()
        assert embed_dim % num_heads == 0, 'El embedding debe ser divisible por el número de cabezas'
        self.num_heads = num_heads
        self.d_head = embed_dim // num_heads
        # Proyecciones Q, K, V y la mezcla final de cabezas concatenadas
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, mask=None, return_attention=False):
        # x: (batch, seq_len, embed_dim)
        B, L, E = x.size()
        # 1) Proyectamos la entrada a Query, Key y Value
        q = self.q_proj(x).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        # -> cada una queda (batch, num_heads, seq_len, d_head)

        # 2) Scaled dot-product attention (por cabeza)
        logits = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        # 3) Máscara de padding: los tokens rellenados no pueden recibir atención
        if mask is not None:
            logits = logits.masked_fill(mask[:, None, None, :] == 0, -1e9)
        attention = torch.softmax(logits, dim=-1)
        values = torch.matmul(attention, v)

        # 4) Concatenamos las cabezas de vuelta a embed_dim y mezclamos
        out = values.transpose(1, 2).contiguous().view(B, L, E)
        out = self.out_proj(out)
        return (out, attention) if return_attention else out

In [ ]:
# Prueba rápida de la atención con una entrada sintética
mha_test = MultiHeadAttention(EMB_DIM, num_heads=8)
x_test = torch.randn(2, 16, EMB_DIM)
mask_test = torch.ones(2, 16, dtype=torch.long)
mask_test[1, 10:] = 0  # la segunda secuencia tiene padding desde la posición 10

out, attn = mha_test(x_test, mask_test, return_attention=True)
print("Salida:", out.shape)          # (2, 16, 128)
print("Atención:", attn.shape)       # (2, 8, 16, 16) = (batch, cabezas, seq, seq)

## **Bloque Transformer (encoder)**

**🎯 Objetivo de la sección:** ensamblar el bloque completo del encoder: multi-head attention + red feed-forward + normalización por capa. **Mejora respecto al notebook guía:** aquí sí usamos **conexiones residuales** (`x + subcapa`), tal como define el paper, lo que estabiliza y acelera el entrenamiento de modelos más profundos.

In [ ]:
class TransformerBlock(nn.Module):
    """Bloque encoder del Transformer con conexiones residuales (como el paper).

    Cada subcapa (atención, FFN) aplica: dropout -> suma residual -> LayerNorm.
    La FFN expande a 4x la dimensión (estándar del paper) y regresa.
    """

    def __init__(self, embed_dim: int, num_heads: int = 8, dropout: float = 0.2):
        super().__init__()
        self.mha = MultiHeadAttention(embed_dim, num_heads)
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
        )
        self.dropout2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, mask=None):
        # 1) Atención multi-cabeza con conexión residual
        attn = self.mha(x, mask)
        x = self.norm1(x + self.dropout1(attn))
        # 2) Red feed-forward con conexión residual
        ffn = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn))
        return x

In [ ]:
# Prueba de humo del bloque completo
block_test = TransformerBlock(EMB_DIM, num_heads=8)
print("Salida del bloque:", block_test(x_test, mask_test).shape)  # (2, 16, 128)

## **Clasificador completo (LightningModule)**

**🎯 Objetivo de la sección:** integrar embeddings + N bloques Transformer + cabeza clasificadora en un `LightningModule`.

**Mejoras respecto al notebook guía:**

1. **Agregación por *mean pooling* con máscara** en lugar del `Flatten` del guía. Aplanar `(batch, 128, 256)` a un vector de 32.768 dimensiones genera una primera capa de ~8M de parámetros (en el guía era de **~134M** con 2048×256); con pooling el vector de entrada a la cabeza es solo de 256 valores y, además, la representación queda alineada con la máscara (el padding no contamina la decisión).
2. **Early stopping sobre `val-acc`** (el guía monitoreaba `train-loss`): nos interesa detenernos cuando la generalización deja de mejorar, no cuando la pérdida de entrenamiento se estanca.
3. **Varios bloques apilables** (`num_layers`) para experimentar con profundidad.

In [ ]:
class ReviewsTransformerClassifier(LightningModule):
    """Clasificador de reseñas con Transformer desde cero.

    Pipeline: embeddings (tokens + posición) -> N bloques transformer ->
    mean pooling con máscara -> cabeza densa de clasificación.
    """

    def __init__(self, vocab_size: int, num_classes: int, max_len: int = 128,
                 emb_dim: int = 256, num_heads: int = 8, num_layers: int = 2,
                 dropout: float = 0.2, pos_type: str = PosEncodingType.SINUSOID,
                 pad_token_id: int = 0, lr: float = 3e-4):
        super().__init__()
        self.save_hyperparameters()

        self.embeddings = TokenAndPosEmbedding(vocab_size, emb_dim, max_len,
                                               pos_type=pos_type, pad_token_id=pad_token_id)
        self.blocks = nn.ModuleList(
            [TransformerBlock(emb_dim, num_heads, dropout) for _ in range(num_layers)]
        )
        self.head = nn.Sequential(
            nn.Linear(emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

        # Métricas de accuracy (una por fase)
        self.train_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.test_acc = Accuracy(task='multiclass', num_classes=num_classes)

    def forward(self, x, mask=None):
        e = self.embeddings(x)
        for block in self.blocks:
            e = block(e, mask)
        # Mean pooling con máscara: promediamos solo los tokens reales
        m = mask.unsqueeze(-1).float()
        pooled = (e * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)
        return self.head(pooled)

    def training_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        loss = F.cross_entropy(y_hat, y)
        self.train_acc(y_hat, y)
        self.log('train-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train-acc', self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        loss = F.cross_entropy(y_hat, y)
        self.val_acc(y_hat, y)
        self.log('val-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val-acc', self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        self.test_acc(y_hat, y)
        self.log('test-acc', self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

    def predict_step(self, batch):
        x, mask = batch['input_ids'], batch['attention_mask']
        return self(x, mask)

    def configure_optimizers(self):
        # AdamW: los modelos desde cero toleran un lr mayor (3e-4) que el
        # fine-tuning de modelos pre-entrenados (2e-5 del notebook guía)
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)
        return optimizer

### **Entrenamiento**

In [ ]:
# Configuración del modelo (ajustable para los experimentos de la sección final)
CONFIG = dict(
    emb_dim=256 if IN_COLAB else 128,
    num_heads=8,
    num_layers=2,           # profundidad: 2 bloques de encoder
    dropout=0.2,
    pos_type=PosEncodingType.SINUSOID,
    lr=3e-4,
)

model = ReviewsTransformerClassifier(
    vocab_size=len(reviews_tokenizer),
    num_classes=NUM_CLASSES,
    max_len=MAX_LEN,
    pad_token_id=reviews_tokenizer.pad_token_id,
    **CONFIG,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {n_params:,} (el clasificador del guía usaba ~140M gracias a su capa Flatten)")

In [ ]:
tb_logger = TensorBoardLogger('tb_logs', name='ReviewsTransformer')
early_stop = EarlyStopping(monitor='val-acc', mode='max', patience=2)

trainer = Trainer(
    max_epochs=5,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    logger=tb_logger,
    callbacks=[early_stop],
    precision='16-mixed' if IN_COLAB else '32-true',  # menos memoria en GPU
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Observemos el proceso de entrenamiento en TensorBoard:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs/

## **Evaluación**

**🎯 Objetivo de la sección:** evaluar el modelo final sobre el conjunto de prueba y analizar el comportamiento por clase (no solo la accuracy global), reutilizando el esquema de métricas del Taller 1 para una comparación directa.

In [ ]:
model.eval()
trainer.test(model, test_loader)

In [ ]:
# Predicciones sobre el test set
logits = trainer.predict(model, test_loader)
logits = torch.cat(logits, dim=0)
pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()

# Etiquetas reales (el test_loader no baraja, así que el orden coincide con test_dataset_pt)
true_ids = np.array([test_dataset_pt[i]['y'].item() for i in range(len(test_dataset_pt))])

# Reporte de clasificación por clase
print(classification_report(true_ids, pred_ids,
                            target_names=[f'{i+1}★' for i in range(NUM_CLASSES)],
                            digits=3))

# Matriz de confusión
cm = confusion_matrix(true_ids, pred_ids)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'{i+1}★' for i in range(NUM_CLASSES)],
            yticklabels=[f'{i+1}★' for i in range(NUM_CLASSES)])
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de confusión - Transformer')
plt.show()

**Análisis (completar tras ejecutar):** en la matriz de confusión, verifica si se repite el patrón del Taller 1 con la LSTM: la mayoría de los errores debería caer en la diagonal adyacente (confundir 2★ con 3★, 3★ con 4★, etc.), lo que confirma que el modelo "entiende" la escala ordinal del sentimiento, aunque falle en el matiz fino entre clases vecinas. Las clases intermedias son las más difíciles porque mezclan aspectos positivos y negativos en la misma reseña.

### **Predicciones y análisis de errores**

**🎯 Objetivo de la sección:** inspeccionar cualitativamente los casos mal clasificados: es donde aprendemos qué le cuesta al modelo.

In [ ]:
df_test = pd.DataFrame({
    'texto': [test_data[i]['text'] for i in range(len(test_data))],
    'categoria': true_ids,
    'prediccion': pred_ids,
})
df_test['correcto'] = df_test['categoria'] == df_test['prediccion']
print(f"Correctas: {df_test['correcto'].sum()} / {len(df_test)} ({df_test['correcto'].mean() * 100:.2f}%)")

errors = df_test[~df_test['correcto']]
print(f"Errores: {len(errors)}")
errors.head(15)

**Análisis de errores (completar tras ejecutar):** revisa si los errores se concentran en (1) reseñas con sentimiento *mixto* ("barato, pero se demoró mucho"), (2) sarcasmo o ironía, y (3) estrellas intermedias. Un ejercicio interesante del notebook guía es contrastar el fallo del modelo con lo que un humano inferiría leyendo solo la primera oración.

## **Comparación con los modelos LSTM del Taller 1**

**🎯 Objetivo de la sección:** comparar la arquitectura Transformer (este notebook) contra los tres modelos LSTM del Taller 1, sobre **los mismos datos, el mismo split y la misma tarea**. Completa la tabla con tus resultados medidos.

| Modelo | Arquitectura | Accuracy (test) | F1 macro (test) |
|---|---|---|---|
| Taller 1 — Modelo 1 | Embeddings + LSTM (simple) | _completar_ | _completar_ |
| Taller 1 — Modelo 2 | LSTM óptima (packing + lengths) | _completar_ | _completar_ |
| Taller 1 — Modelo 3 | LSTM bidireccional | _completar_ | _completar_ |
| **Taller 2 — Transformer** | 2 bloques encoder + PE sinusoidal + mean pooling | _completar_ | _completar_ |

**Hipótesis previas a verificar con tus números:**

- En un corpus corto y con vocabulario moderado, la LSTM ya era fuerte; la ventaja del Transformer puede ser modesta en accuracy pero visible en la **calidad de la representación** (convergencia por época) y en la capacidad de escalar (profundidad, más datos).
- Si el Transformer no supera a la LSTM, revisa: número de épocas (los transformers desde cero suelen necesitar más), tamaño del embedding, y si el early stopping actuó demasiado pronto.
- Un transformer de 1 solo bloque con mean pooling tiene muchos menos parámetros que la cabeza `Flatten` del guía: menos parámetros también es menos overfitting en un corpus de 50k reseñas.

## **Experimentos adicionales (ir más allá del ejemplo)**

**🎯 Objetivo de la sección:** comparar variantes de la arquitectura con un entrenamiento corto y controlado. La celda siguiente entrena dos modelos mínimos (1 bloque, `max_len=64`, subconjunto de 8.000 reseñas, 1 época) que se diferencian **solo** en el tipo de positional encoding (sinusoidal vs. aprendido) e imprime sus accuracies de validación.

Otros experimentos sugeridos (basta cambiar `CONFIG` y re-entrenar):

- `num_layers`: 1 vs 2 vs 3 bloques (¿la profundidad ayuda en textos tan cortos?)
- `num_heads`: 4 vs 8 (¿importa el número de cabezas con `emb_dim=256`?)
- `max_len`: 64 vs 128 (¿cuánta precisión se pierde truncando?)
- `dropout`: 0.1 vs 0.3

In [ ]:
# Micro-experimento: sinusoid vs learnable PE (entrenamiento corto, a igualdad de condiciones)
def entrenar_rapido(pos_type: str, epochs: int = 1, n_train: int = 8000):
    """Entrena una versión mínima del modelo y devuelve su accuracy de validación."""
    pl.seed_everything(42)
    seq_q = 64
    sub_train = Subset(AmazonReviewsTorchDataset(train_data, reviews_tokenizer, seq_q), range(n_train))
    loader_q = DataLoader(sub_train, batch_size=64 if IN_COLAB else 8, shuffle=True)
    val_q = DataLoader(AmazonReviewsTorchDataset(val_data, reviews_tokenizer, seq_q),
                       batch_size=64 if IN_COLAB else 8)
    model_q = ReviewsTransformerClassifier(
        vocab_size=len(reviews_tokenizer), num_classes=NUM_CLASSES,
        max_len=seq_q, emb_dim=128, num_heads=4, num_layers=1,
        pos_type=pos_type, pad_token_id=reviews_tokenizer.pad_token_id,
    )
    trainer_q = Trainer(max_epochs=epochs, logger=False, enable_checkpointing=False,
                           accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1)
    trainer_q.fit(model_q, train_dataloaders=loader_q, val_dataloaders=val_q)
    metrics = trainer_q.validate(model_q, dataloaders=val_q)
    return metrics[0]['val-acc']


for pos_type in (PosEncodingType.SINUSOID, PosEncodingType.LEARNABLE):
    acc = entrenar_rapido(pos_type)
    print(f"PE {pos_type:9s} -> val-acc = {acc:.4f}")

**Cómo leer el micro-experimento:** con 1 época y 8.000 ejemplos la diferencia entre encodings suele ser pequeña; lo interesante es comparar la velocidad de convergencia (curvas de TensorBoard) y repetir el experimento con 2-3 épocas. Documenta en tu entrega qué variante usaste para el modelo final y por qué.

# **Conclusiones**

_(Completar con los resultados obtenidos. Puntos de partida:_

- Comparamos el **mismo problema del Taller 1** (satisfacción en e-commerce, 5 clases, mismos datos y splits) resuelto ahora con un **Transformer construido desde cero**: tokenizador BPE propio, positional embeddings, multi-head attention, bloques encoder con conexiones residuales y agregación por mean pooling.
- El diseño con *mean pooling* con máscara redujo el tamaño del modelo en **más de un orden de magnitud** respecto al enfoque `Flatten` del notebook guía (~9M vs ~140M parámetros), sin sacrificar capacidad de clasificación.
- Las máscaras de atención resuelven de forma elegante el problema del padding que en el Taller 1 requirió `padding_idx` y `pack_padded_sequence`: el padding simplemente no participa en la atención ni en la agregación.
- La tokenización BPE elimina el problema de los tokens desconocidos (`[UNK]`) del vocabulario word-level del Taller 1 y es robusta a errores ortográficos frecuentes en reseñas reales.
- El Transformer requiere más recursos por época (atención cuadrática), pero con textos cortos el costo es bajo y el entrenamiento es paralelizable a diferencia de la LSTM.)

## **Líneas de Trabajo Futuro**

- **Fine-tuning de un modelo pre-entrenado** (BERT en español: `dccuchile/bert-base-spanish-wwm-cased`): los transformers desde cero con 50k ejemplos tienen un techo; el modelo pre-entrenado ya "sabe" el idioma y solo se adapta a la tarea (tema de las próximas sesiones).
- **Clasificación jerárquica o ordinal:** las 5 estrellas son un problema *ordinal*; una pérdida ordinal (por ejemplo, regresión sobre el número de estrellas o penalizar más la confusión de extremos) podría mejorar las matrices de confusión.
- **Análisis de atención:** extraer los `attention` de los casos mal clasificados para ver a qué palabras presta atención el modelo (interpretabilidad).
- **Augmentación de datos** con reseñas paráfraseadas o back-translation para las clases intermedias.
- **Combinação de modelos:** ensamble LSTM + Transformer promediando probabilidades.